In [1]:
import dspy
from datasets import load_dataset

local_llm = dspy.LM(
    "openai/unsloth/Qwen3-Next-80B-A3B-Thinking-GGUF:Q4_K_M", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=0.1,
    cache=False,
)

dspy.configure(lm=local_llm)

In [2]:
class TextToSQL(dspy.Signature):
    """
    Übersetzt eine natürlichsprachliche Frage in eine SQL-Abfrage basierend auf dem gegebenen Schema.
    """
    context = dspy.InputField(desc="Das Schema der Datenbank (Tabellen, Spalten, Typen).")
    question = dspy.InputField(desc="Die Frage, die mittels SQL beantwortet werden soll.")
    sql_query = dspy.OutputField(desc="Die generierte SQL-Abfrage.")

In [3]:
class SQLGenerator(dspy.Module):
    def __init__(self):
        super().__init__()
        # ChainOfThought fördert die logische Herleitung der Query
        self.generate = dspy.ChainOfThought(TextToSQL)

    def forward(self, question, context):
        return self.generate(question=question, context=context)

In [4]:
# Definition des Datenbank-Kontexts
db_schema = """
Tabelle: users
- id (INTEGER)
- name (VARCHAR)
- signup_date (DATE)
- country (VARCHAR)

Tabelle: orders
- id (INTEGER)
- user_id (INTEGER, Foreign Key zu users.id)
- amount (DECIMAL)
- order_date (DATE)
"""

# Trainingsbeispiele erstellen
train_examples = [
    dspy.Example(
        context=db_schema,
        question="Zeige alle Nutzer aus Deutschland.",
        sql_query="SELECT * FROM users WHERE country = 'Deutschland';"
    ).with_inputs('context', 'question'),

    dspy.Example(
        context=db_schema,
        question="Wie hoch ist der Gesamtumsatz aller Bestellungen?",
        sql_query="SELECT SUM(amount) FROM orders;"
    ).with_inputs('context', 'question'),

    dspy.Example(
        context=db_schema,
        question="Liste die Namen der Nutzer auf, die mehr als 100 Euro ausgegeben haben.",
        sql_query="SELECT T1.name FROM users AS T1 JOIN orders AS T2 ON T1.id = T2.user_id WHERE T2.amount > 100;"
    ).with_inputs('context', 'question')
]

In [5]:
from dspy.teleprompt import BootstrapFewShot

# Eigene Metrik definieren: Vergleicht die generierte SQL-Query mit dem Beispiel
def validate_sql(example, pred, trace=None):
    # Einfacher String-Vergleich (bereinigt um Whitespace)
    return example.sql_query.strip() == pred.sql_query.strip()

# Konfiguration des Sprachmodells (Beispiel)
# lm = dspy.LM('openai/gpt-3.5-turbo')
# dspy.settings.configure(lm=lm)

# Initialisierung des Optimizers mit der benutzerdefinierten Metrik
optimizer = BootstrapFewShot(metric=validate_sql)

# Kompilierung des Programms
# Der Optimizer nutzt nun 'validate_sql', um die besten Prompts zu finden.
compiled_sql_generator = optimizer.compile(SQLGenerator(), trainset=train_examples)

100%|████████████████████████████████████████████████████████████████████████████████████████| 3/3 [08:53<00:00, 177.96s/it]

Bootstrapped 1 full traces after 2 examples for up to 1 rounds, amounting to 3 attempts.


In [6]:
# Test des kompilierten Modells
test_question = "Wann hat sich der Nutzer mit der ID 42 angemeldet?"

prediction = compiled_sql_generator(question=test_question, context=db_schema)

print(f"Frage: {test_question}")

# KORREKTUR: Sicherer Zugriff mit getattr, falls 'rationale' fehlt
print(f"Gedankengang: {getattr(prediction, 'rationale', 'Keine Erklärung generiert')}")

print(f"SQL-Query: {prediction.sql_query}")

Frage: Wann hat sich der Nutzer mit der ID 42 angemeldet?
Gedankengang: Keine Erklärung generiert
SQL-Query: SELECT signup_date FROM users WHERE id = 42;


In [7]:
# Optional: Inspektion der Gedankengänge
local_llm.inspect_history(n=10)





[2025-12-08T07:50:39.682158]

System message:

Your input fields are:
1. `context` (str): Das Schema der Datenbank (Tabellen, Spalten, Typen).
2. `question` (str): Die Frage, die mittels SQL beantwortet werden soll.
Your output fields are:
1. `reasoning` (str): 
2. `sql_query` (str): Die generierte SQL-Abfrage.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## sql_query ## ]]
{sql_query}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Übersetzt eine natürlichsprachliche Frage in eine SQL-Abfrage basierend auf dem gegebenen Schema.


User message:

This is an example of the task, though some input or output fields are not supplied.

[[ ## context ## ]]

Tabelle: users
- id (INTEGER)
- name (VARCHAR)
- signup_date (DATE)
- country (VARCHAR)

Tabelle: orders
- id (INTEGER)
- user_id (INTEGER, Foreign 